# Avance 1 – Segmentación de clientes · Pastelería Dulce Tentación
Dataset: Pasteleria_Ventas_Completo.csv

## 1. Carga y vistazo

In [ ]:
# Librerías: pandas/numpy para datos, matplotlib para gráficos y sklearn para el modelo
import pandas as pd, numpy as np, json, os
import matplotlib
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# Rutas: se lee el CSV y los gráficos se guardan en la carpeta img/
BASE = "."  # carpeta donde están el notebook y el CSV
IMG = os.path.join(BASE, "img"); os.makedirs(IMG, exist_ok=True)
df = pd.read_csv(os.path.join(BASE, "Pasteleria_Ventas_Completo.csv"), parse_dates=["Fecha"])
# Diccionario donde se guardan los resultados que se usan en las diapositivas
R = {}  # resultados para las diapositivas
R["shape"] = df.shape
R["dtypes"] = df.dtypes.astype(str).value_counts().to_dict()
print("Forma del dataset (filas, columnas):", df.shape)

# ---- estilo de gráficos (tonos de la presentación) ----
BG, INK, MUTED, GRID = "#FBF6EE", "#3B2620", "#7A6A62", "#E6DCCF"
ONE = "#7A3E2E"           # serie única (cacao)
CAT = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4"]  # validado (CVD OK)
plt.rcParams.update({
    "figure.facecolor": BG, "axes.facecolor": BG, "savefig.facecolor": BG,
    "axes.edgecolor": GRID, "axes.labelcolor": MUTED, "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": .8, "axes.axisbelow": True,
    "axes.spines.top": False, "axes.spines.right": False, "font.size": 13,
    "axes.titlesize": 15, "axes.titleweight": "bold", "axes.titlecolor": INK, "axes.titlelocation": "left",
    "font.family": "DejaVu Sans"})
# Función auxiliar: ajusta, guarda el gráfico como PNG y lo muestra
def save(fig, name):
    fig.tight_layout(); fig.savefig(f"{IMG}/{name}.png", dpi=150); plt.show()

### Primer vistazo: `.head()`

In [ ]:
# Muestra las primeras 5 filas para ver cómo vienen los datos
df.head()

### Tipos de datos: `.info()`

In [ ]:
# Tipo de dato de cada columna y cuántos valores no nulos tiene
df.info()

### Estadísticas descriptivas: `.describe()`

In [ ]:
# Estadísticas básicas (media, desviación, mínimo, máximo, cuartiles) de las columnas numéricas
df.describe()

## 2. Calidad de datos: nulos y duplicados

In [ ]:
# Porcentaje de nulos por columna (solo las que tienen alguno)
nulos = df.isnull().mean().mul(100).sort_values(ascending=False)
nulos = nulos[nulos > 0]
# Guardamos cantidad y % de nulos, y contamos duplicados (todas las columnas iguales excepto el ID)
R["nulos"] = {k: [int(df[k].isnull().sum()), round(v, 1)] for k, v in nulos.items()}
R["duplicados"] = int(df.duplicated(subset=[c for c in df.columns if c != "VentaID"]).sum())
# Gráfico de barras horizontales con el % de nulos y una línea en el umbral de 20%
fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(nulos.index[::-1], nulos.values[::-1], color=ONE, height=.6)
for i, v in enumerate(nulos.values[::-1]):
    ax.text(v + .3, i, f"{v:.1f}%", va="center", color=INK, fontsize=12)
ax.axvline(20, color=MUTED, ls="--", lw=1.2); ax.text(20.3, len(nulos) - .6, "umbral 20%", color=MUTED, fontsize=11)
ax.set_title("Porcentaje de valores nulos por columna"); ax.set_xlabel("% de registros nulos"); ax.grid(axis="y", visible=False)
save(fig, "07_nulos")

## 3. Limpieza

In [ ]:
# Trabajamos sobre una copia para no modificar los datos originales
dfc = df.copy()
# 0) Duplicados: misma venta registrada dos veces (todas las columnas iguales salvo el ID) -> se eliminan
dfc = dfc.drop_duplicates(subset=[c for c in dfc.columns if c != "VentaID"])
R["filas_limpias"] = len(dfc)
# a) Email (22.5% nulos) supera el umbral y no aporta al análisis -> se elimina; Teléfono tampoco aporta
dfc = dfc.drop(columns=["Email", "Telefono"])
# b) Descuento: se reconstruye exactamente con Cantidad*Precio - Monto (regla de negocio)
dfc["Descuento"] = dfc["Descuento"].fillna((dfc.Cantidad * dfc.PrecioUnitario - dfc.MontoTotal).round(2))
# c) CanalVenta: categoría "No registrado" (no se inventa un canal)
dfc["CanalVenta"] = dfc["CanalVenta"].fillna("No registrado")
# d) TipoCliente / Distrito: ffill dentro de cada cliente (el dato existe en otras filas del mismo cliente);
#    lo que quede, con la moda / "Sin dato"
dfc = dfc.sort_values(["NombreCliente", "Fecha"])
for col in ["TipoCliente", "DistritoCliente"]:
    dfc[col] = dfc.groupby("CodigoCliente")[col].transform(lambda s: s.ffill().bfill())
dfc["DistritoCliente"] = dfc["DistritoCliente"].fillna(dfc["DistritoCliente"].mode()[0])
dfc["TipoCliente"] = dfc["TipoCliente"].fillna("Sin dato")
dfc = dfc.drop(columns=["FechaRegistro"])  # no se usa en el modelo
# Verificamos que ya no queden nulos y que Monto = Cantidad*Precio - Descuento
R["nulos_despues"] = int(dfc.isnull().sum().sum())
R["inconsist_monto"] = int(((dfc.Cantidad * dfc.PrecioUnitario - dfc.Descuento - dfc.MontoTotal).abs() > .02).sum())

# Outliers en Cantidad (IQR)
q1, q3 = dfc.Cantidad.quantile([.25, .75]); lim = q3 + 1.5 * (q3 - q1)
R["out_cant"] = [int((dfc.Cantidad > lim).sum()), float(lim), int(dfc.Cantidad.max()),
                 int((dfc.Cantidad >= 80).sum())]

## 4. Serie mensual y retornos (pct_change)

In [ ]:
# Sumamos las ventas por mes (MS = inicio de mes)
mens = dfc.set_index("Fecha").resample("MS").MontoTotal.sum()
# pct_change: variación porcentual de cada mes respecto al anterior
ret = mens.pct_change()
R["mensual"] = {d.strftime("%b-%y"): [round(m), None if np.isnan(r) else round(r * 100, 1)] for d, m, r in zip(mens.index, mens, ret)}
# Etiquetas de los meses en el orden del periodo analizado (nov-2025 a oct-2026)
MES = ["Nov", "Dic", "Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct"]
fig, ax = plt.subplots(figsize=(9, 5))
cols = [ONE if r >= 0 or np.isnan(r) else "#C9A99A" for r in ret.fillna(0)]
ax.bar(MES, ret.fillna(0) * 100, color=["#C9A99A" if (r < 0) else ONE for r in ret.fillna(0)], width=.6)
# Escribimos el % encima (o debajo, si es negativo) de cada barra
for i, r in enumerate(ret):
    if not np.isnan(r):
        ax.text(i, r * 100 + (4 if r >= 0 else -9), f"{r*100:+.0f}%", ha="center", color=INK, fontsize=11)
ax.axhline(0, color=MUTED, lw=1); ax.set_title("Variación mensual de ventas  ·  .pct_change()")
ax.set_ylabel("% vs. mes anterior"); ax.grid(axis="x", visible=False); ax.set_ylim(-65, 135)
save(fig, "08b_pct_change")

## 5. Tabla por cliente (variables para el modelo)

In [ ]:
# Marcamos si la venta fue en fin de semana y tomamos la última fecha para calcular la recencia
dfc["Finde"] = dfc.EsFinDeSemana == 1
fin = dfc.Fecha.max()
# Pasamos de una fila por venta a una fila por cliente, con las variables del modelo:
# frecuencia (días con compra), gasto total, productos, recencia (días desde su última compra)
# y qué proporción de sus compras es en fin de semana, de tortas o de bebidas
cli = dfc.groupby("CodigoCliente").agg(
    Frecuencia=("Fecha", "nunique"),
    MontoTotal=("MontoTotal", "sum"),
    Productos=("Cantidad", "sum"),
    Recencia=("Fecha", lambda s: (fin - s.max()).days),
    PctFinde=("Finde", "mean"),
    PctTortas=("Categoria", lambda s: (s == "Tortas").mean()),
    PctBebidas=("Categoria", lambda s: (s == "Bebidas").mean()),
)
# Ticket promedio = cuánto gasta en promedio cada día que compra
cli["TicketPromedio"] = cli.MontoTotal / cli.Frecuencia
R["n_clientes"] = len(cli)
R["desc_cli"] = cli.describe().round(1).to_dict()

## 6. EDA univariado (4 gráficos)

In [ ]:
# Función para hacer histogramas; con log=True usa escala logarítmica (para datos muy sesgados)
def hist(col, title, xlabel, name, bins=30, log=False):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    data = cli[col]
    if log:
        bins = np.logspace(np.log10(max(data.min(), 1)), np.log10(data.max()), bins); ax.set_xscale("log")
    ax.hist(data, bins=bins, color=ONE, edgecolor=BG, linewidth=1.5)
    ax.axvline(data.median(), color=CAT[1], lw=2, ls="--")
    ax.text(data.median(), ax.get_ylim()[1] * .92, f"  mediana {data.median():,.0f}", color=INK, fontsize=12)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel("N.º de clientes"); ax.grid(axis="x", visible=False)
    save(fig, name)
# Histogramas de las variables principales por cliente
hist("Frecuencia", "Frecuencia de compra (días con compra al año)", "Días con compra", "05a_hist_frecuencia")
hist("MontoTotal", "Monto total gastado por cliente (escala log)", "Soles (S/)", "05b_hist_monto", log=True)
hist("Productos", "Productos adquiridos por cliente (escala log)", "Unidades", "05d_hist_productos", log=True)
# Agrupamos a los clientes por rangos de frecuencia y contamos cuántos hay en cada uno
rangos = pd.cut(cli.Frecuencia, [0, 5, 12, 24, 400], labels=["1–5", "6–12", "13–24", "25+"])
cnt = rangos.value_counts().sort_index()
fig, ax = plt.subplots(figsize=(8, 4.6))
ax.bar(cnt.index.astype(str), cnt.values, color=ONE, width=.55)
for i, v in enumerate(cnt.values): ax.text(i, v + 3, f"{v}  ({v/len(cli):.0%})", ha="center", color=INK, fontsize=12)
ax.set_title("Clientes por rango de frecuencia"); ax.set_xlabel("Días con compra al año"); ax.set_ylabel("N.º de clientes")
ax.grid(axis="x", visible=False); ax.set_ylim(0, cnt.max() * 1.15)
save(fig, "05c_barras_rangos")
R["rangos"] = cnt.to_dict()
# Asimetría (skew): valores altos indican colas largas a la derecha
R["skew"] = cli[["Frecuencia", "MontoTotal", "Productos", "TicketPromedio"]].skew().round(2).to_dict()

## 7. EDA multivariado

In [ ]:
# Función para gráficos de dispersión entre dos variables, mostrando la correlación r en el título
def scatter(x, y, title, xl, yl, name, logx=False, logy=False):
    fig, ax = plt.subplots(figsize=(8, 4.6))
    ax.scatter(cli[x], cli[y], s=22, color=ONE, alpha=.55, edgecolor="none")
    if logx: ax.set_xscale("log")
    if logy: ax.set_yscale("log")
    r = np.corrcoef(np.log1p(cli[x]) if logx else cli[x], np.log1p(cli[y]) if logy else cli[y])[0, 1]
    ax.set_title(f"{title}   (r = {r:.2f})"); ax.set_xlabel(xl); ax.set_ylabel(yl)
    save(fig, name); return round(r, 2)
# Relaciones entre pares de variables
R["r"] = {
    "monto_frec": scatter("Frecuencia", "MontoTotal", "Gasto total vs. frecuencia", "Días con compra", "Gasto total S/ (log)", "06a_sc_gasto_frec", logy=True),
    "prod_monto": scatter("Productos", "MontoTotal", "Productos vs. gasto total", "Unidades (log)", "Gasto total S/ (log)", "06b_sc_prod_gasto", logx=True, logy=True),
    "frec_ticket": scatter("Frecuencia", "TicketPromedio", "Frecuencia vs. ticket promedio", "Días con compra", "Ticket promedio S/ (log)", "06c_sc_frec_ticket", logy=True),
}
# Matriz de correlación entre todas las variables por cliente
vars_corr = ["Frecuencia", "MontoTotal", "Productos", "TicketPromedio", "Recencia", "PctFinde", "PctTortas", "PctBebidas"]
corr = cli[vars_corr].corr()
R["corr"] = corr.round(2).to_dict()
# Mapa de colores azul (negativo) - naranja (positivo) para el heatmap
from matplotlib.colors import LinearSegmentedColormap
div = LinearSegmentedColormap.from_list("div", ["#2a78d6", "#F2EEE8", "#eb6834"])
fig, ax = plt.subplots(figsize=(8.5, 6.4))
im = ax.imshow(corr, cmap=div, vmin=-1, vmax=1)
lab = ["Frecuencia", "Monto", "Productos", "Ticket", "Recencia", "%Finde", "%Tortas", "%Bebidas"]
ax.set_xticks(range(8), lab, rotation=45, ha="right"); ax.set_yticks(range(8), lab); ax.grid(False)
# Escribimos el valor de la correlación dentro de cada celda
for i in range(8):
    for j in range(8):
        ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=10.5,
                color="white" if abs(corr.iloc[i, j]) > .6 else INK)
fig.colorbar(im, ax=ax, shrink=.8); ax.set_title("Matriz de correlación (variables por cliente)")
save(fig, "06d_corr")

## 8. Outliers y transformación

In [ ]:
# Boxplots: a la izquierda los outliers de Cantidad por venta,
# a la derecha el gasto por cliente después de aplicar log1p (se reduce el efecto de los extremos)
fig, axs = plt.subplots(1, 2, figsize=(11, 4.8))
axs[0].boxplot(dfc.Cantidad, vert=False, widths=.5, patch_artist=True,
               boxprops=dict(facecolor="#E8D9CC", color=ONE), medianprops=dict(color=CAT[1], lw=2),
               flierprops=dict(marker="o", markerfacecolor=ONE, markeredgecolor="none", alpha=.5, markersize=6))
axs[0].set_title("Cantidad por venta (antes)"); axs[0].set_yticks([]); axs[0].set_xlabel("Unidades")
axs[1].boxplot(np.log1p(cli.MontoTotal), vert=False, widths=.5, patch_artist=True,
               boxprops=dict(facecolor="#E8D9CC", color=ONE), medianprops=dict(color=CAT[1], lw=2),
               flierprops=dict(marker="o", markerfacecolor=ONE, markeredgecolor="none", alpha=.5, markersize=6))
axs[1].set_title("Gasto por cliente con log1p (después)"); axs[1].set_yticks([]); axs[1].set_xlabel("log(1 + S/)")
save(fig, "08a_boxplots")

## 9. Preparación: log + estandarización

In [ ]:
# Variables que entran al modelo
X = cli[vars_corr].copy()
# log1p a las variables muy sesgadas para que los valores extremos no dominen
for c in ["Frecuencia", "MontoTotal", "Productos", "TicketPromedio"]:
    X[c] = np.log1p(X[c])
R["skew_log"] = X[["Frecuencia", "MontoTotal", "Productos", "TicketPromedio"]].skew().round(2).to_dict()
# Estandarizamos (media 0, desviación 1) para que todas las variables pesen igual en K-Means
Xs = StandardScaler().fit_transform(X)

## 10. PCA

In [ ]:
# PCA: reduce las 8 variables a componentes principales; vemos cuánta varianza explica cada uno
pca = PCA().fit(Xs)
var = pca.explained_variance_ratio_
R["pca_var"] = (var * 100).round(1).tolist()
# Gráfico 1: varianza explicada por componente y acumulada
fig, axs = plt.subplots(1, 2, figsize=(12, 4.8))
axs[0].bar(range(1, 9), var * 100, color=ONE, width=.6)
axs[0].plot(range(1, 9), np.cumsum(var) * 100, color=CAT[1], marker="o", lw=2)
for i, v in enumerate(np.cumsum(var) * 100):
    if i < 4: axs[0].text(i + 1, v + 3, f"{v:.0f}%", ha="center", color=INK, fontsize=11)
axs[0].set_title("Varianza explicada (barras) y acumulada (línea)"); axs[0].set_xlabel("Componente"); axs[0].set_ylabel("%")
axs[0].set_ylim(0, 110); axs[0].grid(axis="x", visible=False)
# Gráfico 2: cargas (loadings), qué variables pesan más en PC1 y PC2
load = pd.DataFrame(pca.components_[:2].T, index=lab, columns=["PC1", "PC2"])
R["loadings"] = load.round(2).to_dict()
y = np.arange(8)
axs[1].barh(y - .2, load.PC1, height=.38, color=CAT[0], label="PC1")
axs[1].barh(y + .2, load.PC2, height=.38, color=CAT[1], label="PC2")
axs[1].set_yticks(y, lab); axs[1].invert_yaxis(); axs[1].axvline(0, color=MUTED, lw=1)
axs[1].set_title("Cargas (loadings) de PC1 y PC2"); axs[1].legend(frameon=False, loc="lower right"); axs[1].grid(axis="y", visible=False)
save(fig, "10_pca")

## 11. K-Means: codo y silhouette

In [ ]:
# Probamos K-Means con k de 2 a 9 y guardamos inercia (codo) y silhouette
ks = range(2, 10); inercia, sil = [], []
for k in ks:
    km = KMeans(k, n_init=20, random_state=42).fit(Xs)
    inercia.append(km.inertia_); sil.append(silhouette_score(Xs, km.labels_))
# Elegimos el k con mayor silhouette (grupos más compactos y separados)
K = int(list(ks)[int(np.argmax(sil))])
R["sil"] = dict(zip(ks, np.round(sil, 3))); R["k"] = K
# Gráficos del método del codo y del silhouette
fig, axs = plt.subplots(1, 2, figsize=(12, 4.6))
axs[0].plot(ks, inercia, color=ONE, marker="o", lw=2); axs[0].axvline(K, color=CAT[1], ls="--")
axs[0].set_title("Método del codo"); axs[0].set_xlabel("k (n.º de grupos)"); axs[0].set_ylabel("Inercia")
axs[1].plot(ks, sil, color=ONE, marker="o", lw=2); axs[1].axvline(K, color=CAT[1], ls="--")
axs[1].text(K + .1, max(sil), f" k = {K}  ·  {max(sil):.2f}", color=INK, fontsize=12, va="top")
axs[1].set_title("Coeficiente silhouette"); axs[1].set_xlabel("k (n.º de grupos)"); axs[1].set_ylabel("Silhouette")
save(fig, "09_codo_silhouette")

# Modelo final con el k elegido; cada cliente recibe su número de grupo
km = KMeans(K, n_init=20, random_state=42).fit(Xs)
cli["Cluster"] = km.labels_
# Perfil promedio de cada grupo para poder interpretarlo
perfil = cli.groupby("Cluster").agg(Clientes=("Frecuencia", "size"), Frecuencia=("Frecuencia", "mean"),
                                     GastoAnual=("MontoTotal", "mean"), Ticket=("TicketPromedio", "mean"),
                                     PctFinde=("PctFinde", "mean"), PctTortas=("PctTortas", "mean"),
                                     PctBebidas=("PctBebidas", "mean"), Recencia=("Recencia", "mean"),
                                     VentasTot=("MontoTotal", "sum"))
# Qué % de las ventas totales aporta cada grupo
perfil["PctVentas"] = perfil.VentasTot / perfil.VentasTot.sum() * 100

# nombre de negocio según el perfil
def nombrar(r):
    if r.Ticket > 300: return "Corporativos"
    if r.PctBebidas > .4 and r.Frecuencia > 25: return "Cafeteros diarios"
    if r.PctTortas > .5: return "Celebraciones"
    if r.PctFinde > .5: return "Familias de fin de semana"
    return "Ocasionales"
perfil["Segmento"] = perfil.apply(nombrar, axis=1)
# Ordenamos los segmentos y asignamos el nombre a cada cliente
orden = ["Cafeteros diarios", "Familias de fin de semana", "Celebraciones", "Ocasionales", "Corporativos"]
perfil = perfil.reset_index().set_index("Segmento").reindex([s for s in orden if s in set(perfil.Segmento)])
print(perfil.round(2))
R["perfil"] = perfil.round(2).reset_index().to_dict(orient="records")
mapa = dict(zip(perfil.Cluster, perfil.index))
cli["Segmento"] = cli.Cluster.map(mapa)

## 12. Visualización de segmentos en el plano PCA

In [ ]:
# Proyectamos a los clientes en las 2 primeras componentes del PCA y los coloreamos por segmento
P = pca.transform(Xs)[:, :2]
fig, ax = plt.subplots(figsize=(10, 6.2))
for i, s in enumerate(perfil.index):
    m = (cli.Segmento == s).values
    ax.scatter(P[m, 0], P[m, 1], s=26, color=CAT[i], alpha=.75, edgecolor=BG, linewidth=.6, label=f"{s} ({m.sum()})")
    cx, cy = P[m, 0].mean(), P[m, 1].mean()
    ax.text(cx, cy, s, fontsize=12, fontweight="bold", color=INK, ha="center",
            bbox=dict(boxstyle="round,pad=.25", fc=BG, ec=CAT[i], lw=1.5))
ax.set_xlabel(f"PC1 · volumen y frecuencia ({var[0]*100:.0f}%)"); ax.set_ylabel(f"PC2 · ({var[1]*100:.0f}%)")
ax.set_title(f"Segmentos K-Means (k = {K}) proyectados en PCA")
ax.legend(frameon=False, loc="upper left", bbox_to_anchor=(1, 1), fontsize=11)
save(fig, "11_segmentos_pca")

# participación en ventas por segmento
fig, ax = plt.subplots(figsize=(9, 4.6))
sc = perfil.sort_values("PctVentas")
ax.barh(sc.index, sc.PctVentas, color=[CAT[list(perfil.index).index(s)] for s in sc.index], height=.6)
for i, (v, n) in enumerate(zip(sc.PctVentas, sc.Clientes)):
    ax.text(v + .6, i, f"{v:.0f}% de ventas · {n/len(cli):.0%} de clientes", va="center", color=INK, fontsize=11.5)
ax.set_xlim(0, sc.PctVentas.max() * 1.6); ax.set_title("Peso de cada segmento en las ventas"); ax.set_xlabel("% de las ventas anuales")
ax.grid(axis="y", visible=False)
save(fig, "12_peso_segmentos")

# Guardamos los clientes con su segmento y los resultados para la presentación
cli.to_csv(os.path.join(BASE, "clientes_segmentados.csv"))
json.dump(R, open(os.path.join(BASE, "resultados.json"), "w"), default=str, indent=1, ensure_ascii=False)
print(json.dumps({k: R[k] for k in ["shape", "nulos", "duplicados", "out_cant", "skew", "skew_log", "r", "pca_var", "sil", "k"]}, default=str, indent=0, ensure_ascii=False))